In [1]:
import pandas as pd

carpeta_entrada = "../02_modelo_canonico/02_segunda_version"
carpeta_salida = "../02_modelo_canonico/03_union"

archivos = {
    "Canonico_1_V2.csv": 245,
    "Canonico_2_V2.csv": 122,
    "Canonico_3_V2.csv": 710,
    "Canonico_4_V2.csv": 178,
    "Canonico_5_V2.csv": 580,
    "Canonico_6_V2.csv": 318,
    "Canonico_ORCID_V2.csv": 429
}

columnas_esperadas = [
    "Fuente_origen",
    "indice",
    "Titulo",
    "Año",
    "Autor_norm",
    "Afiliacion1",
    "Afiliacion2",
    "ISBN",
    "ISSN",
    "Doi",
    "URL",
    "Area",
    "SubArea",
    "Keywords",
    "Abstract"
]

dataframes = []
originales = {}

for archivo, filas_esperadas in archivos.items():

    df = pd.read_csv(
        f"{carpeta_entrada}/{archivo}",
        dtype=str,
        keep_default_na=False
    )

    originales[archivo] = df.copy()

    assert len(df) == filas_esperadas, \
        f"{archivo}: se esperaban {filas_esperadas} filas y hay {len(df)}"

    assert list(df.columns) == columnas_esperadas, \
        f"{archivo}: las columnas no coinciden con el modelo canónico"

    dataframes.append(df)

union = pd.concat(dataframes, ignore_index=True)

assert len(union) == sum(archivos.values())
assert list(union.columns) == columnas_esperadas

# Comprobar que ningún dato original cambió durante la unión
inicio = 0

for archivo, filas_esperadas in archivos.items():

    fin = inicio + filas_esperadas

    bloque = union.iloc[inicio:fin].reset_index(drop=True)

    assert bloque.equals(originales[archivo]), \
        f"Error: se modificaron datos de {archivo}"

    inicio = fin

union.to_csv(
    f"{carpeta_salida}/Canonico_Union_Trabajo.csv",
    index=False,
    encoding="utf-8-sig"
)

# Verificar el archivo que realmente quedó guardado
verificacion = pd.read_csv(
    f"{carpeta_salida}/Canonico_Union_Trabajo.csv",
    dtype=str,
    keep_default_na=False
)

assert verificacion.shape == (2582, 15), \
    f"Error en el archivo guardado: {verificacion.shape}"

assert list(verificacion.columns) == columnas_esperadas, \
    "Error: las columnas del archivo guardado no coinciden"

assert verificacion.equals(union), \
    "Error: el archivo guardado no conserva exactamente los datos de la unión"

for archivo, filas in archivos.items():
    print(f"{archivo}: {filas}")

print("\nTOTAL:", len(verificacion))
print("Columnas:", len(verificacion.columns))
print("Integridad de las 15 columnas originales: OK")
print("Archivo guardado y verificado correctamente.")

Canonico_1_V2.csv: 245
Canonico_2_V2.csv: 122
Canonico_3_V2.csv: 710
Canonico_4_V2.csv: 178
Canonico_5_V2.csv: 580
Canonico_6_V2.csv: 318
Canonico_ORCID_V2.csv: 429

TOTAL: 2582
Columnas: 15
Integridad de las 15 columnas originales: OK
Archivo guardado y verificado correctamente.
